In [1]:
!pip install torch torchvision torchaudio
!pip install datasets
!pip install scikit-learn
!pip install pandas numpy matplotlib

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

from collections import Counter
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd
import re

In [5]:
from datasets import load_dataset

ds = load_dataset("aRWA787/ag_news_dataset")

train_data = ds["train"]
test_data = ds["test"]

print(train_data[0])

{'text': "wall st. bears claw back into the black (reuters) reuters - short-sellers, wall street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2, 'label_name': 'Business', 'length': 144}


In [8]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    return text

In [9]:
train_texts = [clean_text(x['text']) for x in train_data]
train_labels = [x['label'] for x in train_data]

test_texts = [clean_text(x['text']) for x in test_data]
test_labels = [x['label'] for x in test_data]

In [10]:
counter = Counter()

for sentence in train_texts:

    words = sentence.split()

    counter.update(words)

vocab = {'<PAD>':0, '<UNK>':1}

for word, freq in counter.items():

    if freq >= 2:

        vocab[word] = len(vocab)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 50538


In [11]:
MAX_LEN = 100

def text_to_sequence(text):

    words = text.split()

    sequence = []

    for word in words:

        if word in vocab:

            sequence.append(vocab[word])

        else:

            sequence.append(vocab['<UNK>'])

    if len(sequence) < MAX_LEN:

        sequence += [0] * (MAX_LEN - len(sequence))

    else:

        sequence = sequence[:MAX_LEN]

    return sequence

In [12]:
X_train = [text_to_sequence(x) for x in train_texts]
X_test = [text_to_sequence(x) for x in test_texts]

y_train = train_labels
y_test = test_labels

In [13]:
class NewsDataset(Dataset):

    def __init__(self, texts, labels):

        self.texts = texts

        self.labels = labels

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        return (
            torch.tensor(self.texts[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

In [14]:
train_dataset = NewsDataset(X_train, y_train)
test_dataset = NewsDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=64)

In [15]:
class SimpleRNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):

        super(SimpleRNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):

        embedded = self.embedding(x)

        output, hidden = self.rnn(embedded)

        hidden = hidden.squeeze(0)

        out = self.fc(hidden)

        return out

In [16]:
VOCAB_SIZE = len(vocab)

EMBED_DIM = 128

HIDDEN_DIM = 128

OUTPUT_DIM = 4

model = SimpleRNN(
            VOCAB_SIZE,
            EMBED_DIM,
            HIDDEN_DIM,
            OUTPUT_DIM
        )

print(model)

SimpleRNN(
  (embedding): Embedding(50538, 128)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
)


In [17]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
                model.parameters(),
                lr=0.001
            )

In [18]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    correct = 0

    total = 0

    for inputs, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}, Loss={total_loss:.3f}, Accuracy={accuracy:.2f}%"
    )

Epoch 1, Loss=2606.381, Accuracy=25.15%
Epoch 2, Loss=2604.828, Accuracy=25.11%
Epoch 3, Loss=2608.231, Accuracy=24.90%
Epoch 4, Loss=2606.008, Accuracy=24.97%
Epoch 5, Loss=2606.906, Accuracy=25.23%
Epoch 6, Loss=2605.765, Accuracy=25.28%
Epoch 7, Loss=2605.347, Accuracy=25.12%
Epoch 8, Loss=2606.098, Accuracy=25.14%
Epoch 9, Loss=2608.943, Accuracy=25.12%
Epoch 10, Loss=2611.939, Accuracy=25.12%


In [19]:
model.eval()

correct = 0

total = 0

with torch.no_grad():

    for inputs, labels in test_loader:

        outputs = model(inputs)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 26.157894736842106


In [20]:
classes = {

    0:"World",

    1:"Sports",

    2:"Business",

    3:"Sci/Tech"
}

In [21]:
def predict_news(text):

    model.eval()

    text = clean_text(text)

    seq = text_to_sequence(text)

    tensor = torch.tensor(seq).unsqueeze(0)

    with torch.no_grad():

        output = model(tensor)

        prediction = torch.argmax(output, dim=1).item()

    return classes[prediction]

In [22]:
news = "Apple launches new artificial intelligence processor"

category = predict_news(news)

print(category)

Sports
